# BERT-gen — MLM + Gibbs

Историческая генерация из BERT (Wang & Cho, 2019, *BERT has a Mouth*), не causal GPT.

1. **BertMLM** — двусторонний Transformer, post-norm, GELU. В vocab добавлен `[MASK]`.
2. **Обучение** — masked LM: 15% токенов, 80% mask / 10% random / 10% leave. Loss только на масках.
3. **Декод** — сид слева, хвост `[MASK]`, последовательный unmask, затем Gibbs (`n_iters`).

Код: `core/text/bert_gen.py`. Полный train после корпуса:

`python -m training.train_lm --arch bert_gen --level char --d-model 384 --n-layers 8 --n-heads 8 --block-size 256 --steps 45000 --out checkpoints/bert_gen.pt`


In [ ]:
import torch
from core.text.bert_gen import BertMLM, apply_mlm_mask, generate_mlm

text = "Я ебал игру как будто это шлюха\n" * 4
chars = sorted(set(text))
char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for c, i in char_to_idx.items()}
model = BertMLM(len(chars) + 1, d_model=32, n_layers=2, n_heads=4, block_size=32)
ids = torch.tensor([[char_to_idx[c] for c in text[:24]]])
inputs, labels = apply_mlm_mask(ids, model.mask_id, model.vocab_size)
logits = model(inputs)
print(logits.shape, "masked", int((labels != -100).sum()))
print(generate_mlm(model, char_to_idx, idx_to_char, start_text="Я", seq_len=16, n_iters=4, temp=0.9))
